# 🏠 House Prices Prediction – Advanced Regression

This notebook tackles the **House Prices: Advanced Regression Techniques** competition on Kaggle:

🔗 [Competition Link](https://www.kaggle.com/competitions/home-data-for-ml-course)

Due to the relatively small size of the original dataset, we enhanced our training process by incorporating additional data from the **Ames Housing Dataset**:

🔗 [Additional Data Source](https://www.kaggle.com/datasets/prevek18/ames-housing-dataset)

The workflow includes:
- Combining the two datasets for improved model generalization
- Performing EDA and feature selection
- Training a regression model
- Evaluating its performance
- Generating a Kaggle submission file

The goal is to predict house sale prices as accurately as possible using machine learning techniques.


### 📦 Step 1: Import Libraries

This step includes importing all necessary libraries for the project:

- **Data Handling**:  
  `pandas` for working with DataFrames, and `numpy` for numerical operations.

- **Preprocessing & Modeling**:  
  Modules from `scikit-learn` to handle preprocessing (e.g., scaling, encoding), model building, and performance evaluation.

- **Visualization**:  
  `seaborn` and `matplotlib.pyplot` are used to generate plots for data exploration and analysis.


In [ ]:
# Import core libraries for data handling, preprocessing, and modeling
import pandas as pd
import numpy as np

# Import scikit-learn tools for modeling and evaluation
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer

# Import visualization libraries
import seaborn as sns
import matplotlib.pyplot as plt


### 🗂️ Step 2: Load and Explore the Datasets

In this step, we load the three main datasets:

- **train_data**: The original training dataset from the competition.
- **test_data**: The original testing dataset (without target labels).
- **extra_data**: An additional dataset from the *Ames Housing Dataset*, which contains similar house price data.

🔍 **Why additional data?**

The original `train_data` contains only **1,460 rows**, which is considered **very limited** for building robust regression models, especially when using complex techniques like Gradient Boosting or Cross-Validation.

💡 To overcome this limitation and improve generalization, we introduce more data using the `AmesHousing.csv` dataset — a comprehensive dataset that shares the same domain and schema.

📌 **Column Check**: We print the column names of each dataset to ensure that they are compatible and aligned in structure.


In [ ]:
# Load the data
train_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
extra_data = pd.read_csv("/kaggle/input/ames-housing-dataset/AmesHousing.csv")

# Verify the columns are as expected
print("Train data columns: \n", train_data.columns)
print("=="*38)
print("Extra data columns: \n", extra_data.columns)
print("=="*38)
print("Test data columns: \n", test_data.columns)

### 🧪 Step 3: Exploratory Data Analysis (EDA)

In this step, we perform an Exploratory Data Analysis (EDA) to better understand the characteristics of both the `train_data` and the `extra_data` datasets. This analysis includes:

- Displaying the shape (number of rows and columns) of each dataset.
- Checking for missing values in both datasets.
- Reviewing descriptive statistics of the target variable `SalePrice`.
- Visualizing the distribution of `SalePrice` to compare the two datasets.
- Analyzing the data types (numerical vs categorical features).

This step is essential to guide the upcoming data cleaning and preprocessing decisions.


In [ ]:
# Basic shape info
print("📦 Train data shape: ", train_data.shape)
print("📦 Extra data shape: ", extra_data.shape)
print("=="*38)

# Check for missing values
print("\n🔍 Missing values in train_data:")
print(train_data.isnull().sum().sort_values(ascending=False).head(10))

print("\n🔍 Missing values in extra_data:")
print(extra_data.isnull().sum().sort_values(ascending=False).head(10))
print("=="*38)

# Basic statistics for SalePrice
print("\n💲 SalePrice statistics (train_data):")
print(train_data['SalePrice'].describe())

print("\n💲 SalePrice statistics (extra_data):")
print(extra_data['SalePrice'].describe())
print("=="*38)

plt.figure(figsize=(14, 6))
sns.histplot(train_data['SalePrice'], color='blue', label='Train', kde=True)
sns.histplot(extra_data['SalePrice'], color='green', label='Extra', kde=True)
plt.legend()
plt.title("🏠 Distribution of SalePrice in Train vs Extra Data")
plt.xlabel("SalePrice")
plt.ylabel("Frequency")
plt.show()
print("=="*38)

# Types of data
print("\n📋 Feature types (train_data):")
print(train_data.dtypes.value_counts())

print("\n📋 Feature types (extra_data):")
print(extra_data.dtypes.value_counts())


### 🧹 Step 4: Clean and Align Column Names, Then Combine Datasets

In this step, we:

- Standardize all column names (make lowercase and remove spaces/underscores).
- Handle known mismatches like renaming `pid` to `id`, and dropping the `order` column.
- Ensure `saleprice` is consistently named.
- Align both `train_data` and `extra_data` to have the same set of columns.
- Combine both datasets into a single `combined_data`.
- Align `test_data` columns with `combined_data` (excluding `saleprice`) to ensure structure consistency.

> This step ensures that all datasets (`train`, `extra`, `test`) have the same schema,  
> which is necessary before performing preprocessing or training models.


In [ ]:
# Step 0: Define cleaning function
def clean_column_name(col):
    return col.lower().replace(" ", "").replace("_", "")

# Step 1: Clean column names
train_data.columns = [clean_column_name(col) for col in train_data.columns]
extra_data.columns = [clean_column_name(col) for col in extra_data.columns]
test_data.columns = [clean_column_name(col) for col in test_data.columns]

# Step 2: Handle known column mismatches
# Rename PID to id in extra_data
if 'pid' in extra_data.columns:
    extra_data.rename(columns={'pid': 'id'}, inplace=True)

# Drop 'order' column if it exists in extra_data
if 'order' in extra_data.columns:
    extra_data.drop(columns=['order'], inplace=True)

# Ensure SalePrice is consistently lowercase
if 'saleprice' in train_data.columns:
    train_data.rename(columns={'saleprice': 'saleprice'}, inplace=True)
if 'saleprice' in extra_data.columns:
    extra_data.rename(columns={'saleprice': 'saleprice'}, inplace=True)

# Step 3: Get unified column set
all_columns = list(set(train_data.columns).union(set(extra_data.columns)))

# Step 4: Align both dataframes to the same columns
train_data = train_data.reindex(columns=all_columns)
extra_data = extra_data.reindex(columns=all_columns)

# Step 5: Combine the data
combined_data = pd.concat([train_data, extra_data], ignore_index=True)

# Step 6: Align test_data to match combined_data structure (excluding SalePrice)
test_columns = [col for col in combined_data.columns if col != 'saleprice']
test_data = test_data.reindex(columns=test_columns)

# Step 7: Output check
print(f"✅ Final combined_data shape: {combined_data.shape}")
print(f"🧪 Final test_data shape: {test_data.shape}")
print(f"📊 Combined columns: {combined_data.columns.tolist()}")


### 🔄 Step 5: Preprocessing and Data Preparation

In this step, we prepare our data for modeling by:

- Separating the target variable (`saleprice`) from the features.
- Identifying which columns are numerical and which are categorical.
- Defining preprocessing pipelines to handle missing values and encode categorical variables.
- Combining these pipelines using a `ColumnTransformer` to apply the correct transformations to each feature type.
- Applying these preprocessing steps consistently to both the combined training data and the test data.

This ensures that the data is clean, consistent, and ready for model training and prediction.


In [ ]:
# Step 1: Separate the target variable
combined_labels = combined_data['saleprice']
combined_features = combined_data.drop(columns=['saleprice'])

# Step 2: Identify numerical and categorical columns
numeric_cols = combined_features.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = combined_features.select_dtypes(include=['object']).columns.tolist()

# Step 3: Define preprocessing pipelines for numerical and categorical data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

# Step 4: Combine preprocessing steps using ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# Step 5: Apply preprocessing to the data
X_combined_processed = preprocessor.fit_transform(combined_features)
X_test_processed = preprocessor.transform(test_data)

# Display the shapes of the processed datasets
print(f"✅ Processed combined_data shape: {X_combined_processed.shape}")
print(f"🧪 Processed test_data shape: {X_test_processed.shape}")


### 🎯 Steps 6: Feature Selection Based on Correlation

In these steps, we:

- Convert the preprocessed arrays back into DataFrames with meaningful column names.
- Add the target variable (`saleprice`) back to the combined DataFrame.
- Calculate the correlation of each feature with the target variable.
- Identify and display features that have a strong correlation (above 0.5) with the target.
- Select only these highly correlated features to use for training and testing.

This process helps focus the model on the most important features, potentially improving performance and reducing complexity.


In [ ]:
# ✅ Step 6: Convert the processed arrays back to DataFrames with column names
feature_names = preprocessor.get_feature_names_out()
X_combined_df = pd.DataFrame(X_combined_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

# Add SalePrice column back to the combined DataFrame
X_combined_df['saleprice'] = combined_labels.values

# ✅ Step 7: Compute correlation with SalePrice
correlation_matrix = X_combined_df.corr(numeric_only=True)
correlated_features = correlation_matrix['saleprice'][correlation_matrix['saleprice'].abs() > 0.5].drop('saleprice')

# ✅ Step 8: Display highly correlated features
print("🎯 Features with correlation > 0.5 with SalePrice:")
print(correlated_features.sort_values(ascending=False))

# ✅ Step 9: Select only these correlated features from the processed data
selected_features = correlated_features.index.tolist()
X_combined_selected = X_combined_df[selected_features]
X_test_selected = X_test_df[selected_features]


### 🔄 Steps 7: Prepare Data for Modeling

In these steps, we:

- Define the feature matrix `X` and target vector `y` using the selected important features.
- Split the combined dataset into training and validation subsets to evaluate model performance.
- Keep the processed test dataset ready for making final predictions once the model is trained.

This setup ensures proper model training and validation before predicting on unseen data.


In [ ]:
# Step 10: Define features (X) and target (y)
X = X_combined_selected.copy()
y = X_combined_df['saleprice'].copy()

# Step 11: Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 12: X_test_selected will be used for final prediction
print(f"✅ X_train shape: {X_train.shape}")
print(f"✅ X_val shape: {X_val.shape}")
print(f"✅ X_test_selected shape (for prediction): {X_test_selected.shape}")


### 🚀 Model Training and Evaluation

This cell performs the following steps:

1. Initialize a Gradient Boosting Regressor with specified hyperparameters to control learning, complexity, and regularization.
2. Train the model on the training dataset.
3. Make predictions on both the training and validation datasets.
4. Calculate and print the Root Mean Squared Error (RMSE) to evaluate model performance on both sets.

This process helps assess how well the model fits the training data and generalizes to unseen data.


In [ ]:
# 1. Initialize the Gradient Boosting Regressor
model = GradientBoostingRegressor(
    n_estimators=600,
    learning_rate=0.1,
    max_depth=9,
    subsample=0.7,
    min_samples_leaf=15,
    max_features= "sqrt",
    loss='huber',
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42
)

# 2. Train the model
model.fit(X_train, y_train)

# 3. Predict on training set
y_train_pred = model.predict(X_train)

# 4. Predict on validation set
y_val_pred = model.predict(X_val)

# 5. Calculate RMSE for training and validation
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

print(f"Train RMSE: {train_rmse}")
print(f"Valid RMSE: {val_rmse}")


### 📊 Actual vs Predicted Plot

This scatter plot compares actual and predicted values on the validation set. The red dashed line shows perfect predictions, helping visualize how well the model performs.

In [ ]:
sns.scatterplot(x=y_val, y=y_val_pred)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()


### 🏁 Prepare and Save Submission File

In this step, we use the trained model to predict house prices for the test dataset. 

Then, we load the original test file to retrieve the property IDs, combine these IDs with the predicted prices into a DataFrame, and finally save this DataFrame as a CSV file. 

This submission file can be uploaded to the competition platform for evaluation.


In [ ]:
# Step 1: Predict on test set
predictions = model.predict(X_test_selected)

# Optional: Apply np.expm1(predictions) if the target was log-transformed earlier

# Step 2: Load original test file to get the 'Id' column
original_test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# Step 3: Create the submission DataFrame
submission = pd.DataFrame({
    "Id": original_test_df["Id"],
    "SalePrice": predictions
})

# Step 4: Save submission file
submission.to_csv("submission.csv", index=False)

print("✅ Submission file created: submission.csv")


In [ ]:
submission.head()

In [ ]:
submission.describe()